# Corrección con LLM **con contexto del usuario**

Complemento del notebook principal. Pregunta: **¿el LLM corrige mejor si sabe quién es la persona?**
(nombres de su familia, temas de los que habla, su vocabulario). Es lo que un producto como Buendía tiene sobre cada
usuario y un modelo genérico no.

**Cómo lo simulo con S-DiverSe.** No hay perfiles reales de los hablantes. Para cada segmento de **test**, le doy al
LLM, como "lo que sabemos de esta persona", hasta ~350 palabras de lo que **esa misma persona** dijo en sus **otros**
segmentos (transcripción humana). **Nunca** el segmento que se evalúa.

Comparo, con los mismos ejemplos, prompt y salvaguardas:
- **A**: Whisper solo. **A+PP**: con las reglas del paper.
- **B**: A+PP + LLM sin contexto.
- **B+ctx**: A+PP + LLM con contexto del usuario.
- Si el principal ya entrenó C: **D** y **D+ctx** (lo mismo sobre el modelo con fine-tuning).

**Límite honesto de la simulación.** Los segmentos de una persona salen de la misma entrevista, así que el contexto
puede contener frases muy parecidas al segmento evaluado: sería más optimista que un perfil real. Por eso mido el
**solapamiento** entre contexto y segmento, y reporto también el WER **excluyendo** los segmentos con solapamiento alto.

**Requisitos:** que el notebook principal haya generado `resultados/pred_A.csv` y `splits.csv` en Drive (pasa en los
primeros ~20–30 min de su corrida). **No necesita GPU:** usá un entorno de ejecución de CPU.

In [ ]:
%pip install -q "jiwer==4.0.0" num2words anthropic openai

## Configuración
Por defecto OpenAI (`gpt-4.1-mini`, el que usamos). Con Groq, el plan gratuito limita las llamadas por minuto: el
notebook espera y reintenta solo.

In [ ]:
import os, sys, json, re, time, random, zipfile, unicodedata, glob, hashlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

CFG = dict(
    nbest=5,
    llm_proveedor="openai_compatible",  # OpenAI (el que usamos). Otra opción: "anthropic"
    llm_modelo="gpt-4.1-mini",
    llm_esfuerzo="low",
    llm_base_url=None,                  # None = OpenAI; Groq: "https://api.groq.com/openai/v1"
    llm_hilos=8,
    llm_ejemplos=3,
    contexto_palabras=350,            # largo máximo del "perfil" simulado
    seed=42,
)
# Por defecto OpenAI (secreto OPENAI_API_KEY). Para Claude, descomentá y cargá ANTHROPIC_API_KEY:
# CFG.update(llm_proveedor="anthropic", llm_base_url=None, llm_modelo="claude-haiku-5-5", llm_hilos=8)

if "google.colab" in sys.modules and not os.environ.get("SDIVERSE_BASE"):
    from google.colab import drive
    drive.mount("/content/drive")
BASE = os.environ.get("SDIVERSE_BASE", "/content/drive/MyDrive/Shippear/sdiverse_hackathon")
RES = os.path.join(BASE, "resultados")
ZIP_DIR = os.path.dirname(BASE)

## Datos
Leo el TSV directamente del zip (sin descomprimir audios), el split por hablante y las transcripciones de Whisper
que guardó el notebook principal.

In [ ]:
faltan = [p for p in (os.path.join(BASE, "splits.csv"), os.path.join(RES, "pred_A.csv")) if not os.path.exists(p)]
if faltan:
    raise FileNotFoundError(f"Todavía no existen {faltan}. Esperá a que el notebook principal termine la sección 5.")

tsv = None
for z in sorted(glob.glob(os.path.join(ZIP_DIR, "*.zip"))):
    try:
        zf = zipfile.ZipFile(z)
        nombre = [n for n in zf.namelist() if n.endswith("s-diverse.tsv")]
        if nombre:
            tsv = pd.read_csv(zf.open(nombre[0]), sep="\t"); break
    except zipfile.BadZipFile:
        pass
assert tsv is not None, f"No encuentro el zip con s-diverse.tsv en {ZIP_DIR}"

splits = pd.read_csv(os.path.join(BASE, "splits.csv"))
predA = pd.read_csv(os.path.join(RES, "pred_A.csv"), keep_default_na=False)
predA["nbest"] = predA.nbest.map(json.loads)
df = (tsv.merge(splits[["Sample_ID", "split"]], on="Sample_ID")
         .merge(predA.rename(columns={"hyp": "A", "nbest": "A_nbest"}), on="Sample_ID"))
print(f"{len(df)} segmentos | test: {(df.split == 'test').sum()} segmentos de "
      f"{df[df.split == 'test'].Speaker_ID.nunique()} hablantes")

## Funciones compartidas con el notebook principal
Normalización, WER y reglas del paper: **idénticas** al principal (copiadas automáticamente al generar este notebook),
así los números son comparables.

In [ ]:
import jiwer
from num2words import num2words

def _num_a_palabras(m):
    s = m.group(0).replace(".", "")
    try:
        return " " + num2words(int(s), lang="es") + " "
    except Exception:
        return " " + s + " "

def normalizar(texto, quitar_tildes=False):
    t = unicodedata.normalize("NFC", str(texto)).lower()
    t = t.replace("<unk>", " ")
    t = re.sub(r"\d{1,3}(?:\.\d{3})+(?!\d)|\d+", _num_a_palabras, t)
    t = re.sub(r"[^\w\s]|_", " ", t)
    if quitar_tildes:
        t = unicodedata.normalize("NFD", t)
        t = re.sub("[̀́̈]", "", t)   # grave, agudo, diéresis (la virgulilla de la ñ queda)
        t = unicodedata.normalize("NFC", t)
    return " ".join(t.split())

def contar_errores(ref, hyp):
    """Errores de un segmento con jiwer. ref/hyp ya normalizados."""
    r, h = ref.split(), hyp.split()
    if not r:
        return dict(S=0, D=0, I=len(h), N=0)
    if not h:
        return dict(S=0, D=len(r), I=0, N=len(r))
    o = jiwer.process_words(ref, hyp)
    return dict(S=o.substitutions, D=o.deletions, I=o.insertions, N=o.hits + o.substitutions + o.deletions)

def wer_segmentos(tabla, col_hyp, sin_tildes=False):
    """Una fila por segmento con S, D, I, N y WER."""
    filas = []
    for ref, hyp in zip(tabla.Transcription, tabla[col_hyp].fillna("")):
        filas.append(contar_errores(normalizar(ref, sin_tildes), normalizar(hyp, sin_tildes)))
    e = pd.DataFrame(filas, index=tabla.index, columns=["S", "D", "I", "N"])   # columnas fijas: tabla puede venir vacía
    e["WER"] = (e.S + e.D + e.I) / e.N.replace(0, np.nan)
    return e

def resumen_wer(tabla, metodos, por=None, sin_tildes=False):
    """WER agregado (suma de errores / suma de palabras) por grupo, con conteos."""
    tabla = tabla[tabla.Transcription.map(normalizar).str.len() > 0]
    claves = [por] if por else []
    base = tabla.groupby(claves) if por else tabla.assign(_g="TOTAL").groupby("_g")
    out = base.agg(segmentos=("Sample_ID", "size"), hablantes=("Speaker_ID", "nunique"))
    out["palabras"] = tabla.Transcription.map(lambda x: len(normalizar(x).split())).groupby(
        tabla[por] if por else pd.Series("TOTAL", index=tabla.index)).sum()
    for m in metodos:
        if m not in tabla or tabla[m].isna().all():
            continue
        e = wer_segmentos(tabla, m, sin_tildes)
        g = e.groupby(tabla[por] if por else pd.Series("TOTAL", index=tabla.index))
        s = g.sum()
        out[m] = (100 * (s.S + s.D + s.I) / s.N).round(1)
    return out

def _reducir_palabra(w):
    if len(w) <= 15:
        return w
    for L in range(1, len(w) // 2 + 1):
        if (w[:L] * (len(w) // L + 1))[:len(w)] == w:
            return w[:L]
    return w

def _clave(w):
    return re.sub(r"[^\w]", "", w.lower())

def postprocesar_reglas(texto, max_n=40):
    """Las 3 reglas del paper (sección 3.4). Solo borran: nunca agregan palabras."""
    toks = [_reducir_palabra(w) for w in str(texto).split()]
    out = []
    for w in toks:                                         # (ii) palabra repetida
        if out and _clave(w) and _clave(w) == _clave(out[-1]):
            continue
        out.append(w)
    toks, claves = out, [_clave(w) for w in out]
    cambio = True
    while cambio:                                          # (iii) frase repetida
        cambio = False
        for n in range(min(len(toks) // 2, max_n), 1, -1):
            i = 0
            while i + 2 * n <= len(toks):
                if claves[i:i + n] == claves[i + n:i + 2 * n]:
                    del toks[i + n:i + 2 * n]; del claves[i + n:i + 2 * n]; cambio = True
                else:
                    i += 1
    return " ".join(toks)

# Pruebas rápidas (si alguna falla, el notebook se detiene acá)
assert normalizar("¿Esto tiene tratamiento?\" y dice: \"no, no hay\".") == "esto tiene tratamiento y dice no no hay"
assert normalizar("Eso fue en el 20 y en 2009 <unk>") == "eso fue en el veinte y en dos mil nueve"
assert normalizar("Está aquí, señor", quitar_tildes=True) == "esta aqui señor"
assert postprocesar_reglas("y entonces y entonces y entonces me fui me fui") == "y entonces me fui"
assert postprocesar_reglas("jajajajajajajajajaja vale") == "ja vale"
assert contar_errores("a b c", "a x c d") == dict(S=1, D=0, I=1, N=3)
print("normalización, WER y reglas: OK")

## Funciones de LLM (idénticas al principal)
Uso un archivo de caché propio (`cache_llm_contexto.json`) para no pisar el del principal si corren a la vez.

In [ ]:
SYSTEM_LLM = """Corregís transcripciones automáticas (ASR) de habla espontánea en español producida por personas con disartria (ELA, Parkinson o secuelas de ACV). Recibís una o varias hipótesis del reconocedor para el MISMO audio, ordenadas de más a menos probable.

Devolvé la transcripción más probable de lo que la persona dijo.
- Corregí solo errores típicos de reconocimiento: palabras que suenan parecido, palabras inexistentes, cortes o uniones de palabras, repeticiones en bucle del reconocedor.
- Usá las hipótesis alternativas para decidir entre palabras dudosas.
- No agregues información que no esté en ninguna hipótesis, no resumas, no parafrasees y no corrijas la gramática ni el estilo de la persona.
- Mantené muletillas y repeticiones naturales ("eh", "o sea", "no, no").
- Si la primera hipótesis parece correcta, devolvela sin cambios. Ante la duda, cambiá lo menos posible.
- Respondé solo con la transcripción final, sin comillas ni comentarios."""

def obtener_secreto(nombre):
    if os.environ.get(nombre):
        return os.environ[nombre]
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nombre)
    except Exception:
        return None

_cliente = {}
def _llm_anthropic(mensajes):
    import anthropic
    if "a" not in _cliente:
        _cliente["a"] = anthropic.Anthropic(api_key=obtener_secreto("ANTHROPIC_API_KEY"), max_retries=4)
    kw = dict(model=CFG["llm_modelo"], max_tokens=4000, system=SYSTEM_LLM, messages=mensajes)
    if CFG["llm_esfuerzo"]:
        kw["output_config"] = {"effort": CFG["llm_esfuerzo"]}
    if CFG["llm_modelo"] in ("claude-opus-5-5", "claude-opus-5", "claude-fable-5-1", "claude-sonnet-5-5"):
        r = _cliente["a"].beta.messages.create(betas=["server-side-fallback-2026-07-01"], fallbacks="default", **kw)
    else:
        r = _cliente["a"].messages.create(**kw)
    if r.stop_reason == "refusal":
        return None
    return "".join(b.text for b in r.content if b.type == "text")

def nombre_clave():
    if CFG["llm_proveedor"] == "anthropic":
        return "ANTHROPIC_API_KEY"
    return "GROQ_API_KEY" if "groq" in (CFG["llm_base_url"] or "") else "OPENAI_API_KEY"

def _llm_openai(mensajes):
    from openai import OpenAI
    if "o" not in _cliente:
        _cliente["o"] = OpenAI(api_key=obtener_secreto(nombre_clave()) or "sin-clave",
                               base_url=CFG["llm_base_url"], max_retries=0)
    r = _cliente["o"].chat.completions.create(
        model=CFG["llm_modelo"], messages=[{"role": "system", "content": SYSTEM_LLM}] + mensajes)
    texto = r.choices[0].message.content or ""
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.S)   # modelos que razonan en voz alta

ERRORES_DEFINITIVOS = {"NotFoundError", "AuthenticationError", "PermissionDeniedError", "BadRequestError",
                       "UnprocessableEntityError"}

def probar_llm():
    """Una llamada de prueba antes de lanzar todas. Devuelve True si el proveedor y el modelo responden."""
    f = {"anthropic": _llm_anthropic, "openai_compatible": _llm_openai}[CFG["llm_proveedor"]]
    try:
        f([{"role": "user", "content": "Hipótesis 1: hola buenos días"}])
        print(f"LLM OK: {CFG['llm_proveedor']} / {CFG['llm_modelo']}")
        return True
    except Exception as e:
        print(f"El LLM no responde ({type(e).__name__}): {str(e)[:300]}")
        if type(e).__name__ == "NotFoundError":
            print(f"El modelo '{CFG['llm_modelo']}' no existe en el proveedor. Modelos disponibles:")
            try:
                from openai import OpenAI
                if CFG["llm_proveedor"] == "openai_compatible":
                    cli = OpenAI(api_key=obtener_secreto(nombre_clave()), base_url=CFG["llm_base_url"])
                    print(sorted(m.id for m in cli.models.list().data))
            except Exception:
                pass
            print('Elegí uno grande de propósito general y poné CFG["llm_modelo"] = "..." en la configuración.')
        elif type(e).__name__ in ("AuthenticationError", "PermissionDeniedError"):
            print(f"Revisá el secreto {nombre_clave()}: clave inválida o sin 'Acceso al notebook'.")
        return False

RUTA_CACHE = os.path.join(RES, "cache_llm.json")
CACHE_LLM = json.load(open(RUTA_CACHE, encoding="utf-8")) if os.path.exists(RUTA_CACHE) else {}

def llamar_llm(mensajes):
    clave = hashlib.sha1(json.dumps([CFG["llm_proveedor"], CFG["llm_modelo"], SYSTEM_LLM, mensajes],
                                    ensure_ascii=False).encode()).hexdigest()
    if clave in CACHE_LLM:
        return CACHE_LLM[clave]
    f = {"anthropic": _llm_anthropic, "openai_compatible": _llm_openai}[CFG["llm_proveedor"]]
    for intento in range(6):   # reintentos con espera creciente: los planes gratuitos limitan llamadas por minuto
        try:
            salida = f(mensajes)
            break
        except Exception as e:
            # Modelo inexistente, clave inválida o pedido mal formado: reintentar no sirve
            if intento == 5 or type(e).__name__ in ERRORES_DEFINITIVOS:
                raise
            espera = min(60, 5 * 2 ** intento)
            print(f"LLM: {type(e).__name__}, reintento en {espera}s")
            time.sleep(espera)
    CACHE_LLM[clave] = salida
    return salida

def formatear_hipotesis(hyps):
    return "\n".join(f"Hipótesis {i+1}: {h}" for i, h in enumerate(hyps))

def hipotesis_para_llm(greedy, nbest):
    vistas, hyps = set(), []
    for h in [greedy] + list(nbest):
        h = postprocesar_reglas(h)
        if h.strip() and normalizar(h) not in vistas:
            vistas.add(normalizar(h)); hyps.append(h)
    return hyps[:CFG["nbest"]] or [greedy]

def elegir_ejemplos(k):
    """Ejemplos few-shot de hablantes de ENTRENAMIENTO, uno por enfermedad, con errores moderados."""
    tr = df[(df.split == "train") & (df.Duration.between(5, 20))].copy()
    tr["wer"] = wer_segmentos(tr, "A").WER
    tr = tr[tr.wer.between(0.15, 0.6)].sort_values("wer")
    ej = tr.groupby("Disease").head(1).head(k)
    msgs = []
    for _, r in ej.iterrows():
        msgs += [{"role": "user", "content": formatear_hipotesis(hipotesis_para_llm(r.A, r.A_nbest))},
                 {"role": "assistant", "content": r.Transcription.replace("<unk>", "").strip()}]
    return msgs, ej.Sample_ID.tolist()

N_DESCARTADAS = {"n": 0}
def corregir(greedy, nbest, ejemplos):
    hyps = hipotesis_para_llm(greedy, nbest)
    base = hyps[0]
    if not base.strip():
        return base
    try:
        salida = llamar_llm(ejemplos + [{"role": "user", "content": formatear_hipotesis(hyps)}])
    except Exception as e:
        N_DESCARTADAS["fallos"] = N_DESCARTADAS.get("fallos", 0) + 1
        print("Error de LLM:", type(e).__name__, str(e)[:200])
        return base
    salida = (salida or "").strip().strip('"“”«»').strip()
    nb, ns = len(base.split()), len(salida.split())
    if not salida or ns < 0.5 * nb or ns > 1.5 * nb + 3:
        N_DESCARTADAS["n"] += 1
        return base
    return salida

def aplicar_llm(tabla, col_greedy, col_nbest, ejemplos):
    from concurrent.futures import ThreadPoolExecutor
    filas = list(zip(tabla[col_greedy], tabla[col_nbest]))
    N_DESCARTADAS["fallos"] = 0
    with ThreadPoolExecutor(CFG["llm_hilos"]) as ex:
        out = list(tqdm(ex.map(lambda f: corregir(f[0], f[1], ejemplos), filas), total=len(filas), desc="LLM"))
    json.dump(CACHE_LLM, open(RUTA_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    if N_DESCARTADAS["fallos"]:
        print(f"ATENCIÓN: {N_DESCARTADAS['fallos']} de {len(filas)} llamadas fallaron y quedaron SIN corregir. "
              "Volvé a correr la celda: las exitosas salen de la caché y solo se reintentan las fallidas.")
    return out

df["A+PP"] = df["A"].map(postprocesar_reglas)
RUTA_CACHE = os.path.join(RES, "cache_llm_contexto.json")
CACHE_LLM = json.load(open(RUTA_CACHE, encoding="utf-8")) if os.path.exists(RUTA_CACHE) else {}
if not obtener_secreto(nombre_clave()):
    raise RuntimeError(f"Falta el secreto {nombre_clave()} (ícono de llave en Colab).")
if not os.environ.get("SDIVERSE_PRUEBA_LOCAL") and not probar_llm():
    raise RuntimeError("El LLM no responde: corregí CFG (ver el mensaje de arriba) y volvé a correr desde acá.")

## Contexto del usuario (simulado)

Para cada segmento: otros segmentos de la misma persona, en orden aleatorio fijo, hasta `contexto_palabras`. Va al
principio del mensaje con una instrucción explícita: usarlo **solo para resolver palabras dudosas** (nombres
propios, lugares, temas recurrentes) y **no copiar frases** que no estén respaldadas por las hipótesis.

**Solapamiento:** porcentaje de los grupos de 4 palabras seguidas del segmento evaluado que también aparecen en el
contexto. Un valor alto significa que el contexto "regala" parte de la respuesta.

In [ ]:
PREFIJO_CTX = ("Información sobre esta persona, tomada de otras conversaciones suyas. Usala SOLO para resolver "
               "palabras dudosas (nombres propios, lugares, temas recurrentes). No copies frases de acá que no estén "
               "respaldadas por las hipótesis.\n<contexto>\n{ctx}\n</contexto>\n\n")

def construir_contexto(sample_id, speaker_id):
    otros = df[(df.Speaker_ID == speaker_id) & (df.Sample_ID != sample_id)]
    otros = otros.sample(frac=1, random_state=CFG["seed"])
    partes, n = [], 0
    for t in otros.Transcription:
        t = " ".join(t.replace("<unk>", "…").split())
        if n + len(t.split()) > CFG["contexto_palabras"]:
            break
        partes.append(t); n += len(t.split())
    return "\n".join(partes)

def solapamiento(ref, ctx, n=4):
    r, c = normalizar(ref).split(), normalizar(ctx).split()
    g_r = {tuple(r[i:i + n]) for i in range(len(r) - n + 1)}
    g_c = {tuple(c[i:i + n]) for i in range(len(c) - n + 1)}
    return len(g_r & g_c) / len(g_r) if g_r else 0.0

def corregir_ctx(greedy, nbest, ejemplos, contexto):
    """Igual que corregir() del principal, con el contexto antes de las hipótesis."""
    hyps = hipotesis_para_llm(greedy, nbest)
    base = hyps[0]
    if not base.strip():
        return base
    contenido = (PREFIJO_CTX.format(ctx=contexto) if contexto else "") + formatear_hipotesis(hyps)
    try:
        salida = llamar_llm(ejemplos + [{"role": "user", "content": contenido}])
    except Exception as e:
        N_DESCARTADAS["fallos"] = N_DESCARTADAS.get("fallos", 0) + 1
        print("Error de LLM:", type(e).__name__, str(e)[:200])
        return base
    salida = (salida or "").strip().strip('"“”«»').strip()
    nb, ns = len(base.split()), len(salida.split())
    if not salida or ns < 0.5 * nb or ns > 1.5 * nb + 3:
        N_DESCARTADAS["n"] += 1
        return base
    return salida

def aplicar(tabla, col, col_nbest, ejemplos, con_contexto):
    from concurrent.futures import ThreadPoolExecutor
    N_DESCARTADAS["fallos"] = 0
    filas = [(r[col], r[col_nbest], r.ctx if con_contexto else "") for _, r in tabla.iterrows()]
    with ThreadPoolExecutor(CFG["llm_hilos"]) as ex:
        out = list(tqdm(ex.map(lambda f: corregir_ctx(f[0], f[1], ejemplos, f[2]), filas), total=len(filas),
                        desc=f"{col} {'con' if con_contexto else 'sin'} contexto"))
    json.dump(CACHE_LLM, open(RUTA_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    if N_DESCARTADAS["fallos"]:
        print(f"ATENCIÓN: {N_DESCARTADAS['fallos']} llamadas fallaron y quedaron SIN corregir. "
              "Volvé a correr la celda: las exitosas salen de la caché.")
    return out

TEST = df[df.split == "test"].copy()
TEST["ctx"] = [construir_contexto(s, k) for s, k in zip(TEST.Sample_ID, TEST.Speaker_ID)]
TEST["solap"] = [solapamiento(r, c) for r, c in zip(TEST.Transcription, TEST.ctx)]
print(f"Contexto promedio: {TEST.ctx.str.split().str.len().mean():.0f} palabras | "
      f"solapamiento medio {100*TEST.solap.mean():.1f}% | segmentos con solapamiento > 20%: {(TEST.solap > 0.2).sum()}")
print("\nEjemplo de contexto para", TEST.iloc[0].Sample_ID, ":\n", TEST.iloc[0].ctx[:600], "...")

## B vs B+ctx en test

In [ ]:
EJEMPLOS, ids_ej = elegir_ejemplos(CFG["llm_ejemplos"])
assert not set(ids_ej) & set(TEST.Sample_ID)
print("Ejemplos few-shot (train):", ids_ej)
TEST["B"] = aplicar(TEST, "A", "A_nbest", EJEMPLOS, con_contexto=False)
TEST["B+ctx"] = aplicar(TEST, "A", "A_nbest", EJEMPLOS, con_contexto=True)
print(f"Respuestas descartadas por la salvaguarda de largo: {N_DESCARTADAS['n']}")

ruta_C = os.path.join(RES, "pred_C.csv")
if os.path.exists(ruta_C):
    pc = pd.read_csv(ruta_C, keep_default_na=False)
    pc["nbest"] = pc.nbest.map(json.loads)
    TEST = TEST.merge(pc.rename(columns={"hyp": "C", "nbest": "C_nbest"}), on="Sample_ID", how="left")
    if TEST.C.notna().all():
        TEST["C+PP"] = TEST["C"].map(postprocesar_reglas)
        TEST["D"] = aplicar(TEST, "C", "C_nbest", EJEMPLOS, con_contexto=False)
        TEST["D+ctx"] = aplicar(TEST, "C", "C_nbest", EJEMPLOS, con_contexto=True)
else:
    print("El principal todavía no generó pred_C.csv: comparo solo sobre Whisper sin entrenar.")

METODOS = [m for m in ["A", "A+PP", "B", "B+ctx", "C", "C+PP", "D", "D+ctx"] if m in TEST]
for nombre, por in [("total", None), ("por enfermedad", "Disease"), ("por hablante", "Speaker_ID")]:
    print(f"\n=== WER (%) en TEST, {nombre} ===")
    display(resumen_wer(TEST, METODOS, por))

bajo = TEST[TEST.solap <= 0.2]
print(f"\n=== Robustez: solo segmentos con solapamiento <= 20% ({len(bajo)} de {len(TEST)}) ===")
display(resumen_wer(bajo, METODOS))
TEST.drop(columns=[c for c in TEST if c.endswith("nbest")]).to_csv(
    os.path.join(RES, "transcripciones_contexto.csv"), index=False)

## ¿Qué cambió el contexto?
Segmentos donde B y B+ctx difieren. Fijate si las mejoras son nombres propios y temas (lo esperado) o frases
copiadas del contexto (señal de que la simulación es optimista).

In [ ]:
t = TEST[TEST["B"].map(normalizar) != TEST["B+ctx"].map(normalizar)].copy()
t["WER B"] = wer_segmentos(t, "B").WER
t["WER B+ctx"] = wer_segmentos(t, "B+ctx").WER
print(f"El contexto cambió {len(t)} de {len(TEST)} segmentos: mejoró {(t['WER B+ctx'] < t['WER B']).sum()}, "
      f"empeoró {(t['WER B+ctx'] > t['WER B']).sum()}.")
for _, r in t.assign(g=t["WER B"] - t["WER B+ctx"]).sort_values("g", ascending=False).head(8).iterrows():
    print(f"\n[{r.Sample_ID}] WER {100*r['WER B']:.0f}% → {100*r['WER B+ctx']:.0f}%  (solapamiento {100*r.solap:.0f}%)")
    print("  REF   :", r.Transcription[:250]); print("  B     :", r.B[:250]); print("  B+ctx :", r["B+ctx"][:250])

## Para la demo: corregir con un perfil escrito a mano
Así se ve el producto: el perfil es lo que la app sabe del usuario. Probá con una transcripción de Whisper y un perfil
inventado, por ejemplo el de la diapositiva ("ama a ti" → "llamá a Lucía").

In [ ]:
def corregir_con_perfil(texto_whisper, perfil, ejemplos=EJEMPLOS):
    return corregir_ctx(texto_whisper, [], ejemplos, perfil)

perfil = "Se llama Marta, tiene ELA. Su hija se llama Lucía y su marido Andrés. Vive en Valencia y le gusta la jardinería."
for frase in ["ama a ti que venga", "dile a andres que me traiga las tijeras del jardín"]:
    print(f"Whisper: {frase}\nCon perfil: {corregir_con_perfil(frase, perfil)}\n")